## 1. 🧱 1. Imports & setup

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    r2_score,
    root_mean_squared_error,
    mean_absolute_error,
)
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [2]:
random_state = 42
test_size = 0.2

## 📊 2. Chargement des données

In [3]:
data = fetch_california_housing(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=test_size, random_state=random_state
)

print(X_train.shape, X_test.shape)

(16512, 8) (4128, 8)


## 🧪 3. Baseline : Régression linéaire

In [14]:
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

y_pred = lin_model.predict(X_test)

lin_metrics = {
            "R2": r2_score(y_train, lin_model.predict(X_train)),
            "RMSE": root_mean_squared_error(y_test, y_pred),
            "MAE": mean_absolute_error(y_test, y_pred),
        }

print(f"Linear Regression R2: {lin_metrics['R2']:.4f}")
print(f"Linear Regression RMSE: {lin_metrics['RMSE']:.4f}")
print(f"Linear Regression MAE: {lin_metrics['MAE']:.4f}")

Linear Regression R2: 0.6126
Linear Regression RMSE: 0.7456
Linear Regression MAE: 0.5332


## 🌲 4. Random Forest avec GridSearch

In [7]:
rf = RandomForestRegressor(random_state=random_state)

param_grid_rf = {
    "n_estimators": [100, 120],
    "max_depth": [5, 10, None],
    "max_features": [3, None],
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

print("Best RF params:", grid_rf.best_params_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best RF params: {'max_depth': None, 'max_features': 3, 'n_estimators': 120}


### 📉 Évaluation

In [15]:
y_pred_rf = best_rf.predict(X_test)
rf_metrics = {
            "R2": r2_score(y_train, best_rf.predict(X_train)),
            "RMSE": root_mean_squared_error(y_test, y_pred_rf),
            "MAE": mean_absolute_error(y_test, y_pred_rf),
        }

print(f"Random Forest R2: {rf_metrics['R2']:.4f}")
print(f"Random Forest RMSE: {rf_metrics['RMSE']:.4f}")
print(f"Random Forest MAE: {rf_metrics['MAE']:.4f}")

Random Forest R2: 0.9750
Random Forest RMSE: 0.4936
Random Forest MAE: 0.3232


🚀 5. Gradient Boosting avec GridSearch

In [20]:
gb = GradientBoostingRegressor(random_state=random_state)

param_grid_gb = {
    "learning_rate": [0.1, 0.2, 0.3],
    "n_estimators": [100, 120],
    "max_depth": [5, None],
    "max_features": [None, 3],
}

grid_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid_gb,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

grid_gb.fit(X_train, y_train)

best_gb = grid_gb.best_estimator_

print("Best GB params:", grid_gb.best_params_)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best GB params: {'learning_rate': 0.2, 'max_depth': 5, 'max_features': None, 'n_estimators': 120}


### 📉 Évaluation

In [21]:
y_pred_gb = best_gb.predict(X_test)
gb_metrics = {
            "R2": r2_score(y_train, best_gb.predict(X_train)),
            "RMSE": root_mean_squared_error(y_test, y_pred_gb),
            "MAE": mean_absolute_error(y_test, y_pred_gb),
        }

print(f"Gradient Boosting R2: {gb_metrics['R2']:.4f}")
print(f"Gradient Boosting RMSE: {gb_metrics['RMSE']:.4f}")
print(f"Gradient Boosting MAE: {gb_metrics['MAE']:.4f}")

Gradient Boosting R2: 0.9091
Gradient Boosting RMSE: 0.4759
Gradient Boosting MAE: 0.3167


## 📊 6. Comparaison finale

In [22]:
# Create a DataFrame to compare metrics
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Gradient Boosting'],
    'R2': [lin_metrics['R2'], rf_metrics['R2'], gb_metrics['R2']],
    'RMSE': [lin_metrics['RMSE'], rf_metrics['RMSE'], gb_metrics['RMSE']],
    'MAE': [lin_metrics['MAE'], rf_metrics['MAE'], gb_metrics['MAE']]
})

results.sort_values(by='RMSE', inplace=True)

print(results)

               Model        R2      RMSE       MAE
2  Gradient Boosting  0.909127  0.475854  0.316749
1      Random Forest  0.974992  0.493635  0.323247
0  Linear Regression  0.612551  0.745581  0.533200


## 🏆 7. Sélection du meilleur modèle : Conclusion

Parmi les modèles testés, le modèle **Gradient Boosting** est retenu.

Bien que le modèle Random Forest présente un R² plus élevé (0.97),
le Gradient Boosting offre de meilleures performances sur les métriques
d'erreur :

- RMSE plus faible (0.475 vs 0.493)
- MAE plus faible (0.316 vs 0.323)

Ces métriques étant plus représentatives de l'erreur réelle de prédiction,
le modèle Gradient Boosting *{'learning_rate': 0.2, 'max_depth': 5, 'max_features': None, 'n_estimators': 120}* est considéré comme plus précis pour ce problème.

Le modèle retenu sera donc utilisé pour l'entraînement final et le logging avec MLflow.